# EM for PCA on face images

[pcaImageDemo.ipynb](pcaImageDemo.ipynb) (figure 20.3) computes the eigenfaces of the Olivetti faces with an SVD.
This notebook computes them with EM instead (PML book 1, section 20.2.5; Roweis, 1998) and checks that it gives the
same eigenfaces, eigenvalues and reconstructions.

In [ ]:
import time

import numpy as np
from matplotlib import pyplot as plt
from scipy.linalg import subspace_angles
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA

try:
    import probml_utils as pml
except ModuleNotFoundError:
    %pip install -qq git+https://github.com/probml/probml-utils.git
    import probml_utils as pml

## Data

400 images of $64 \times 64$ pixels: $N = 400$, $D = 4096$.

In [ ]:
faces = fetch_olivetti_faces()
X = faces["data"]
N, D = X.shape
h, w = 64, 64
mu = X.mean(axis=0)
Xc = X - mu
print(f"N = {N}, D = {D}")

## EM for PCA

EM for PCA is EM for PPCA in the limit $\sigma^2 \to 0$. With the centred data $\tilde{\mathbf{X}}$ ($N \times D$) and
$\mathbf{W}$ ($D \times L$):

- **E step:** project the data onto the current subspace, $\mathbf{Z} = (\mathbf{W}^\top \mathbf{W})^{-1} \mathbf{W}^\top \tilde{\mathbf{X}}^\top$ ($L \times N$);
- **M step:** refit the subspace to those projections, $\mathbf{W} = \tilde{\mathbf{X}}^\top \mathbf{Z}^\top (\mathbf{Z} \mathbf{Z}^\top)^{-1}$.

Each iteration costs $O(NDL)$ and never forms the $D \times D$ covariance. EM converges to the principal subspace, but
$\mathbf{W}$ is some basis of it, not the eigenvectors. They come from a small last step: orthonormalise $\mathbf{W}$ to
$\mathbf{Q}$ and diagonalise the $L \times L$ covariance of the projections $\tilde{\mathbf{X}} \mathbf{Q}$.

In [ ]:
def pca_em(Xc, L, n_iter=1000, tol=1e-8, seed=0):
    W = np.random.default_rng(seed).standard_normal((Xc.shape[1], L))
    rmse = []
    for it in range(n_iter):
        Z = np.linalg.solve(W.T @ W, W.T @ Xc.T)  # E step, [L, N]
        W = Xc.T @ Z.T @ np.linalg.inv(Z @ Z.T)  # M step, [D, L]
        Q, _ = np.linalg.qr(W)
        rmse.append(np.sqrt(np.mean((Xc - Xc @ Q @ Q.T) ** 2)))
        if it > 0 and rmse[-2] - rmse[-1] < tol * rmse[-1]:
            break
    # eigenvectors and eigenvalues inside the subspace, largest first
    evals, evecs = np.linalg.eigh(np.cov((Xc @ Q).T))
    V = (Q @ evecs[:, ::-1]).T  # [L, D], rows are eigenfaces
    return V, evals[::-1], np.array(rmse)


t = time.perf_counter()
pca = PCA().fit(Xc)
print(f"sklearn PCA, all {pca.n_components_} components: {time.perf_counter() - t:.2f} s")

Ls = [5, 20, 50]
em = {}
for L in Ls:
    t = time.perf_counter()
    V_em, ev_em, rmse = pca_em(Xc, L)
    dt = time.perf_counter() - t
    V_em *= np.sign(np.sum(V_em * pca.components_[:L], axis=1, keepdims=True))  # the sign of an eigenvector is arbitrary
    em[L] = V_em, ev_em, rmse
    V = pca.components_[:L]
    print(
        f"L = {L:2d}: {len(rmse):3d} iterations, {dt:.2f} s; "
        f"largest angle to the PCA subspace {np.degrees(subspace_angles(V.T, V_em.T).max()):.2f} deg, "
        f"smallest eigenface cosine {np.sum(V * V_em, axis=1).min():.4f}, "
        f"largest relative eigenvalue error {np.max(np.abs(ev_em / pca.explained_variance_[:L] - 1)):.0e}"
    )

## Eigenfaces

The mean face and the first five eigenfaces from PCA (top) and from EM with $L = 20$ (bottom).

In [ ]:
V_em = em[20][0]
fig, axs = plt.subplots(2, 6, figsize=(10, 4))
for row, (V, name) in enumerate([(pca.components_, "PCA"), (V_em, "EM")]):
    axs[row, 0].imshow(mu.reshape(h, w), cmap="gray")
    axs[row, 0].set_title("mean")
    for i in range(5):
        axs[row, i + 1].imshow(V[i].reshape(h, w), cmap="gray")
        axs[row, i + 1].set_title(f"{name} {i + 1}")
for ax in axs.flat:
    ax.axis("off")
fig.tight_layout()
pml.savefig("pcaImageEmEigenfaces.pdf")
plt.show()

## Reconstructions

Image 125 (the one [pcaImageDemo.ipynb](pcaImageDemo.ipynb) uses) reconstructed from 5, 10 and 20 components.

In [ ]:
ndx = 125
Ks = [5, 10, 20]
fig, axs = plt.subplots(2, 4, figsize=(8, 4.4))
for row, (V, name) in enumerate([(pca.components_, "PCA"), (V_em, "EM")]):
    axs[row, 0].imshow(X[ndx].reshape(h, w), cmap="gray")
    axs[row, 0].set_title("original")
    for i, k in enumerate(Ks):
        recon = Xc[ndx] @ V[:k].T @ V[:k] + mu
        axs[row, i + 1].imshow(recon.reshape(h, w), cmap="gray")
        axs[row, i + 1].set_title(f"{name}, K = {k}")
for ax in axs.flat:
    ax.axis("off")
fig.tight_layout()
pml.savefig("pcaImageEmRecon.pdf")
plt.show()

for k in Ks:
    print(f"K = {k:2d}: max pixel difference between PCA and EM reconstructions "
          f"{np.abs(Xc[ndx] @ pca.components_[:k].T @ pca.components_[:k] - Xc[ndx] @ V_em[:k].T @ V_em[:k]).max():.1e}")

## Convergence

Reconstruction RMSE of all 400 images per EM iteration; the dashed lines are the PCA values, the lowest RMSE
any $L$-dimensional subspace can reach.

In [ ]:
colors = ["#2a78d6", "#eb6834", "#1baf7a"]
fig, ax = plt.subplots(figsize=(6, 4))
for L, color in zip(Ls, colors):
    rmse = em[L][2]
    V = pca.components_[:L]
    ax.plot(np.arange(1, len(rmse) + 1), rmse, color=color, lw=2, label=f"EM, L = {L}")
    ax.axhline(np.sqrt(np.mean((Xc - Xc @ V.T @ V) ** 2)), color=color, ls="--", lw=1)
ax.set_xscale("log")
ax.set_xlabel("iteration")
ax.set_ylabel("reconstruction RMSE")
ax.legend()
fig.tight_layout()
pml.savefig("pcaImageEmConvergence.pdf")
plt.show()

EM recovers the same eigenfaces, eigenvalues and reconstructions as the SVD.

- **Speed:** EM reaches the PCA error within a few iterations, then spends most of its iterations on the last
  digits; how many it needs depends on the gap between the $L$-th and $(L+1)$-th eigenvalues, not on $L$.
- **Cost:** here the SVD is faster: with only 400 images it is cheap, and it gives all components at once.
  EM pays off when $D$ and $N$ are both large and only a few components are needed, or when data are missing.